# 📱 Liquidity Stress Prediction Challenge - Starter Notebook

Welcome to the my notebook for this challenge.

In this challenge, my goal is to build a machine learning model that predicts whether a customer is likely to experience liquidity stress, using historical mobile money behaviour.

This notebook walks through a simple end-to-end pipeline:

- Loading and exploring the data
- Preprocessing
- Training a baseline model
- Evaluating performance
- Generating predictions for submission

The baseline model used here is **Logistic Regression**.


In [1]:
# 📚 Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)


In [2]:
# 📥 Load the Data
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

# Peek at the data
train.head()


,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highe

## Data cleaning and exploration

Here we inspect the data for important information that could affect the training process and model building.

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Columns: 184 entries, arpu to ID
dtypes: float64(92), int64(86), object(6)
memory usage: 56.2+ MB


In [4]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Columns: 183 entries, arpu to ID
dtypes: float64(92), int64(85), object(6)
memory usage: 41.9+ MB


In [5]:
# Check missing values
missing_summary = train.isnull().sum().sort_values(ascending=False)
# missing_summary[missing_summary > 0].head(20)
missing_summary ## There are no missing values

arpu                                    0
m6_received_highest_amount              0
m4_received_total_value                 0
m4_received_highest_amount              0
m4_received_senders                     0
                                       ..
m2_transfer_from_bank_banks             0
m3_transfer_from_bank_volume            0
m3_transfer_from_bank_total_value       0
m3_transfer_from_bank_highest_amount    0
ID                                      0
Length: 184, dtype: int64

In [6]:
missing_summary[missing_summary > 0]

Series([], dtype: int64)

There are no missing values.

## Feature Engineering
Here we will create new columns that could give more meaning to the features. For example means, volatility and trends can be gotten from the monthly data. This will create a clear understanding of the patterns or overall behaviours of the candidates transactions.

In [7]:
train.columns

Index(['arpu', 'age', 'gender', 'region', 'smartphone', 'segment',
       'earning_pattern', 'x_90_d_activity_rate', 'm1_paybill_volume',
       'm1_paybill_total_value',
       ...
       'm6_withdraw_highest_amount', 'm6_withdraw_agents', 'm1_daily_avg_bal',
       'm2_daily_avg_bal', 'm3_daily_avg_bal', 'm4_daily_avg_bal',
       'm5_daily_avg_bal', 'm6_daily_avg_bal', 'liquidity_stress_next_30d',
       'ID'],
      dtype='object', length=184)

In [8]:
## features of concern
features_to_engineer=[
    'paybill_volume','paybill_total_value','paybill_highest_amount','paybill_companies',
    'merchantpay_volume','merchantpay_total_value','merchantpay_highest_amount','merchantpay_merchants',
    'transfer_from_bank_volume','transfer_from_bank_total_value','transfer_from_bank_highest_amount','transfer_from_bank_banks',
    'mm_send_volume','mm_send_total_value','mm_send_highest_amount','mm_send_recipients',
    'received_volume','received_total_value','received_highest_amount','received_senders',
    'deposit_volume','deposit_total_value','deposit_highest_amount','deposit_agents',
    'withdraw_volume','withdraw_total_value','withdraw_highest_amount','withdraw_agents',   
]

feat=['m1_daily_avg_bal','m2_daily_avg_bal','m3_daily_avg_bal','m4_daily_avg_bal','m5_daily_avg_bal','m6_daily_avg_bal']

In [9]:
# EPS = 1e-6
# T = np.array([6, 5, 4, 3, 2, 1], dtype=float)   # M1 (recent) = 6 ... M6 (old) = 1
# TC = T - T.mean()
# DENOM = TC @ TC

# def _slope(x):
#     return (x - x.mean(axis=1, keepdims=True)) @ TC / DENOM

# def add_trend_features(df):
#     new = {}
#     for f in features_to_engineer:
#         cols = [f"m{i}_{f}" for i in range(1, 7)]
#         v = df[cols].fillna(0)                      # NaN month = no activity
#         mean = v.mean(axis=1)
#         new[f"{f}_mean"] = mean
#         new[f"{f}_std"] = v.std(axis=1)
#         recent = v[cols[:3]].mean(axis=1)
#         older = v[cols[3:]].mean(axis=1)
#         new[f"{f}_ratio"] = (recent + EPS) / (older + EPS)

#         if f.endswith(("_volume", "_total_value")):
#             s = _slope(v.to_numpy(dtype=float))
#             new[f"{f}_slope"] = s
#             new[f"{f}_rel_slope"] = s / (mean.to_numpy() + EPS)

#     bal = [f"m{i}_daily_avg_bal" for i in range(1, 7)]
#     b = df[bal]
#     bal_mean = b.mean(axis=1)
#     new["bal_mean"] = bal_mean
#     new["bal_std"] = b.std(axis=1)
#     new["bal_ratio"] = (b[bal[:3]].mean(axis=1) + EPS) / (b[bal[3:]].mean(axis=1) + EPS)
#     bs = _slope(b.to_numpy(dtype=float))
#     new["bal_slope"] = bs
#     new["bal_rel_slope"] = bs / (bal_mean.to_numpy() + EPS)

#     return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

# train_engineered = add_trend_features(train)
# test_engineered = add_trend_features(test)

We will engineer new features to derive more meaning in the features eg inflow/outflow, active months etc

In [10]:
import numpy as np
import pandas as pd

# category -> its counterparty column suffix
CATS = {
    "paybill": "companies", "merchantpay": "merchants", "transfer_from_bank": "banks",
    "mm_send": "recipients", "received": "senders", "deposit": "agents", "withdraw": "agents",
}
INFLOW = ["transfer_from_bank", "received", "deposit"]
OUTFLOW = ["paybill", "merchantpay", "mm_send", "withdraw"]

EPS = 1e-6
T = np.array([6, 5, 4, 3, 2, 1], dtype=float)   # M1 (recent) = 6 ... M6 (old) = 1
TC = T - T.mean()
DENOM = TC @ TC


def _arr(df, cat, metric):
    """(n, 6) array for M1..M6, NaN month treated as no activity."""
    cols = [f"m{i}_{cat}_{metric}" for i in range(1, 7)]
    return df[cols].fillna(0).to_numpy(dtype=float)


def _slope(x):
    return (x - x.mean(axis=1, keepdims=True)) @ TC / DENOM


def _recent_older(x):
    return (x[:, :3].mean(axis=1) + EPS) / (x[:, 3:].mean(axis=1) + EPS)


def add_features(df):
    new = {}

    # 1. Per-category trend features (same as before)
    for cat, cp in CATS.items():
        for metric in ["volume", "total_value", "highest_amount", cp]:
            x = _arr(df, cat, metric)
            f = f"{cat}_{metric}"
            mean = x.mean(axis=1)
            new[f"{f}_mean"] = mean
            new[f"{f}_std"] = x.std(axis=1, ddof=1)
            new[f"{f}_ratio"] = _recent_older(x)
            if metric in ("volume", "total_value"):
                s = _slope(x)
                new[f"{f}_slope"] = s
                new[f"{f}_rel_slope"] = s / (mean + EPS)

    # 2. Balance features (NaN months left as NaN)
    b = df[[f"m{i}_daily_avg_bal" for i in range(1, 7)]]
    bal = b.to_numpy(dtype=float)
    bal_mean = b.mean(axis=1).to_numpy()
    new["bal_mean"] = bal_mean
    new["bal_std"] = b.std(axis=1).to_numpy()
    new["bal_ratio"] = ((b.iloc[:, :3].mean(axis=1) + EPS) / (b.iloc[:, 3:].mean(axis=1) + EPS)).to_numpy()
    bs = _slope(bal)
    new["bal_slope"] = bs
    new["bal_rel_slope"] = bs / (bal_mean + EPS)

    # 3. Cash-flow features
    val = {c: _arr(df, c, "total_value") for c in CATS}
    inflow = sum(val[c] for c in INFLOW)
    outflow = sum(val[c] for c in OUTFLOW)
    net = inflow - outflow

    for name, x in [("inflow", inflow), ("outflow", outflow)]:
        new[f"{name}_mean"] = x.mean(axis=1)
        new[f"{name}_std"] = x.std(axis=1, ddof=1)
        new[f"{name}_ratio"] = _recent_older(x)
        new[f"{name}_slope"] = _slope(x)

    new["net_mean"] = net.mean(axis=1)
    new["net_std"] = net.std(axis=1, ddof=1)
    new["net_slope"] = _slope(net)
    new["net_recent_minus_older"] = net[:, :3].mean(axis=1) - net[:, 3:].mean(axis=1)
    new["neg_net_months"] = (net < 0).sum(axis=1)
    new["neg_net_months_last3"] = (net[:, :3] < 0).sum(axis=1)

    new["out_in_ratio_6m"] = outflow.sum(axis=1) / (inflow.sum(axis=1) + EPS)
    new["out_in_ratio_last3"] = outflow[:, :3].sum(axis=1) / (inflow[:, :3].sum(axis=1) + EPS)
    new["inflow_cv"] = inflow.std(axis=1, ddof=1) / (inflow.mean(axis=1) + EPS)
    new["inflow_months"] = (inflow > 0).sum(axis=1)

    # runway: how many months of spending the balance covers
    runway = bal / (outflow + EPS)
    new["runway_m1"] = runway[:, 0]
    new["runway_mean"] = pd.DataFrame(runway).mean(axis=1).to_numpy()
    new["bal_vs_mean_outflow"] = bal_mean / (outflow.mean(axis=1) + EPS)
    max_withdrawal = _arr(df, "withdraw", "highest_amount").max(axis=1)
    new["bal_vs_max_withdrawal"] = bal[:, 0] / (max_withdrawal + EPS)

    new["cash_out_share"] = val["withdraw"].sum(axis=1) / (inflow.sum(axis=1) + EPS)
    new["deposit_to_withdraw"] = val["deposit"].sum(axis=1) / (val["withdraw"].sum(axis=1) + EPS)
    new["paybill_share_outflow"] = val["paybill"].sum(axis=1) / (outflow.sum(axis=1) + EPS)

    # 4. Activity, 5. size and concentration (per category)
    for cat, cp in CATS.items():
        vol = _arr(df, cat, "volume")
        active = vol > 0
        new[f"{cat}_active_months"] = active.sum(axis=1)
        new[f"{cat}_active_last3"] = active[:, :3].sum(axis=1)
        new[f"{cat}_zero_streak"] = np.cumprod((~active).astype(int), axis=1).sum(axis=1)

        v = val[cat]
        hi = _arr(df, cat, "highest_amount")
        cpv = _arr(df, cat, cp)
        new[f"{cat}_avg_ticket"] = v.sum(axis=1) / (vol.sum(axis=1) + EPS)
        new[f"{cat}_spike"] = hi.max(axis=1) / (v.mean(axis=1) + EPS)
        new[f"{cat}_txn_per_counterparty"] = vol.mean(axis=1) / (cpv.mean(axis=1) + EPS)

    # 6. Profile interactions
    arpu = df["arpu"].to_numpy(dtype=float)
    age = df["age"].to_numpy(dtype=float)
    new["arpu_to_bal"] = arpu / (np.abs(bal_mean) + EPS)
    new["bal_per_age"] = bal_mean / (age + 1)
    new["arpu_to_outflow"] = arpu / (outflow.mean(axis=1) + EPS)

    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)


# 7. Group-relative features: fit medians on train, apply everywhere (no target used)
GROUP_COLS = ["region", "segment"]
GROUP_FEATS = ["bal_mean", "outflow_mean", "inflow_mean"]


def fit_group_stats(df):
    return {(g, f): df.groupby(g)[f].median() for g in GROUP_COLS for f in GROUP_FEATS}


def add_group_features(df, stats):
    new = {}
    for (g, f), med in stats.items():
        new[f"{f}_vs_{g}"] = df[f] / (df[g].map(med).abs() + EPS)
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)


# usage (run BEFORE one-hot encoding, since region/segment must still be raw)
train_engineered = add_features(train)
test_engineered = add_features(test)
group_stats = fit_group_stats(train_engineered)
train_engineered_n = add_group_features(train_engineered, group_stats)
test_engineered_n = add_group_features(test_engineered, group_stats)

In [11]:
train_engineered_n.shape

(40000, 377)

In [12]:
train_engineered_n.head()

,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,...,deposit_agents_mean,deposit_agents_std,deposit_agents_ratio,withdraw_volume_mean,withdraw_volume_std,withdraw_volume_ratio,withdraw_volume_slope,withdraw_volume_rel_slope,withdraw_total_value_mean,withdraw_total_value_std,withdraw_total_value_ratio,withdraw_total_value_slope,withdraw_total_value_rel_slope,withdraw_highest_amount_mean,withdraw_highest_amount_std,withdraw_highest_amount_ratio,withdraw_agents_mean,withdraw_agents_std,withdraw_agents_ratio,bal_mean,bal_std,bal_ratio,bal_slope,bal_rel_slope,inflow_mean,inflow_std,inflow_ratio,inflow_slope,outflow_mean,outflow_std,outflow_ratio,outflow_slope,net_mean,net_std,net_slope,net_recent_minus_older,neg_net_months,neg_net_months_last3,out_in_ratio_6m,out_in_ratio_last3,inflow_cv,inflow_months,runway_m1,runway_mean,bal_vs_mean_outflow,bal_vs_max_withdrawal,cash_out_share,deposit_to_withdraw,paybill_share_outflow,paybill_active_months,paybill_active_last3,paybill_zero_streak,paybill_avg_ticket,paybill_spike,paybill_txn_per_counterparty,merchantpay_active_months,merchantpay_active_last3,merchantpay_zero_streak,merchantpay_avg_ticket,merchantpay_spike,merchantpay_txn_per_counterparty,transfer_from_bank_active_months,transfer_from_bank_active_last3,transfer_from_bank_zero_streak,transfer_from_bank_avg_ticket,transfer_from_bank_spike,transfer_from_bank_txn_per_counterparty,mm_send_active_months,mm_send_active_last3,mm_send_zero_streak,mm_send_avg_ticket,mm_send_spike,mm_send_txn_per_counterparty,received_active_months,received_active_last3,received_zero_streak,received_avg_ticket

In [13]:
features_old = [
    'm1_paybill_volume', 'm1_paybill_total_value', 'm1_paybill_highest_amount', 'm1_paybill_companies',
    'm2_paybill_volume', 'm2_paybill_total_value', 'm2_paybill_highest_amount', 'm2_paybill_companies',
    'm3_paybill_volume', 'm3_paybill_total_value', 'm3_paybill_highest_amount', 'm3_paybill_companies',
    'm4_paybill_volume', 'm4_paybill_total_value', 'm4_paybill_highest_amount', 'm4_paybill_companies',
    'm5_paybill_volume', 'm5_paybill_total_value', 'm5_paybill_highest_amount', 'm5_paybill_companies',
    'm6_paybill_volume', 'm6_paybill_total_value', 'm6_paybill_highest_amount', 'm6_paybill_companies',
    'm1_merchantpay_volume', 'm1_merchantpay_total_value', 'm1_merchantpay_highest_amount', 'm1_merchantpay_merchants',
    'm2_merchantpay_volume', 'm2_merchantpay_total_value', 'm2_merchantpay_highest_amount', 'm2_merchantpay_merchants',
    'm3_merchantpay_volume', 'm3_merchantpay_total_value', 'm3_merchantpay_highest_amount', 'm3_merchantpay_merchants',
    'm4_merchantpay_volume', 'm4_merchantpay_total_value', 'm4_merchantpay_highest_amount', 'm4_merchantpay_merchants',
    'm5_merchantpay_volume', 'm5_merchantpay_total_value', 'm5_merchantpay_highest_amount', 'm5_merchantpay_merchants',
    'm6_merchantpay_volume', 'm6_merchantpay_total_value', 'm6_merchantpay_highest_amount', 'm6_merchantpay_merchants',
    'm1_transfer_from_bank_volume', 'm1_transfer_from_bank_total_value', 'm1_transfer_from_bank_highest_amount', 'm1_transfer_from_bank_banks',
    'm2_transfer_from_bank_volume', 'm2_transfer_from_bank_total_value', 'm2_transfer_from_bank_highest_amount', 'm2_transfer_from_bank_banks',
    'm3_transfer_from_bank_volume', 'm3_transfer_from_bank_total_value', 'm3_transfer_from_bank_highest_amount', 'm3_transfer_from_bank_banks',
    'm4_transfer_from_bank_volume', 'm4_transfer_from_bank_total_value', 'm4_transfer_from_bank_highest_amount', 'm4_transfer_from_bank_banks',
    'm5_transfer_from_bank_volume', 'm5_transfer_from_bank_total_value', 'm5_transfer_from_bank_highest_amount', 'm5_transfer_from_bank_banks',
    'm6_transfer_from_bank_volume', 'm6_transfer_from_bank_total_value', 'm6_transfer_from_bank_highest_amount', 'm6_transfer_from_bank_banks',
    'm1_mm_send_volume', 'm1_mm_send_total_value', 'm1_mm_send_highest_amount', 'm1_mm_send_recipients',
    'm2_mm_send_volume', 'm2_mm_send_total_value', 'm2_mm_send_highest_amount', 'm2_mm_send_recipients',
    'm3_mm_send_volume', 'm3_mm_send_total_value', 'm3_mm_send_highest_amount', 'm3_mm_send_recipients',
    'm4_mm_send_volume', 'm4_mm_send_total_value', 'm4_mm_send_highest_amount', 'm4_mm_send_recipients',
    'm5_mm_send_volume', 'm5_mm_send_total_value', 'm5_mm_send_highest_amount', 'm5_mm_send_recipients',
    'm6_mm_send_volume', 'm6_mm_send_total_value', 'm6_mm_send_highest_amount', 'm6_mm_send_recipients',
    'm1_received_volume', 'm1_received_total_value', 'm1_received_highest_amount', 'm1_received_senders',
    'm2_received_volume', 'm2_received_total_value', 'm2_received_highest_amount', 'm2_received_senders',
    'm3_received_volume', 'm3_received_total_value', 'm3_received_highest_amount', 'm3_received_senders',
    'm4_received_volume', 'm4_received_total_value', 'm4_received_highest_amount', 'm4_received_senders',
    'm5_received_volume', 'm5_received_total_value', 'm5_received_highest_amount', 'm5_received_senders',
    'm6_received_volume', 'm6_received_total_value', 'm6_received_highest_amount', 'm6_received_senders',
    'm1_deposit_volume', 'm1_deposit_total_value', 'm1_deposit_highest_amount', 'm1_deposit_agents',
    'm2_deposit_volume', 'm2_deposit_total_value', 'm2_deposit_highest_amount', 'm2_deposit_agents',
    'm3_deposit_volume', 'm3_deposit_total_value', 'm3_deposit_highest_amount', 'm3_deposit_agents',
    'm4_deposit_volume', 'm4_deposit_total_value', 'm4_deposit_highest_amount', 'm4_deposit_agents',
    'm5_deposit_volume', 'm5_deposit_total_value', 'm5_deposit_highest_amount', 'm5_deposit_agents',
    'm6_deposit_volume', 'm6_deposit_total_value', 'm6_deposit_highest_amount', 'm6_deposit_agents',
    'm1_withdraw_volume', 'm1_withdraw_total_value', 'm1_withdraw_highest_amount', 'm1_withdraw_agents',
    'm2_withdraw_volume', 'm2_withdraw_total_value', 'm2_withdraw_highest_amount', 'm2_withdraw_agents',
    'm3_withdraw_volume', 'm3_withdraw_total_value', 'm3_withdraw_highest_amount', 'm3_withdraw_agents',
    'm4_withdraw_volume', 'm4_withdraw_total_value', 'm4_withdraw_highest_amount', 'm4_withdraw_agents',
    'm5_withdraw_volume', 'm5_withdraw_total_value', 'm5_withdraw_highest_amount', 'm5_withdraw_agents',
    'm6_withdraw_volume', 'm6_withdraw_total_value', 'm6_withdraw_highest_amount', 'm6_withdraw_agents',
    'm1_daily_avg_bal', 'm2_daily_avg_bal', 'm3_daily_avg_bal', 'm4_daily_avg_bal', 'm5_daily_avg_bal', 'm6_daily_avg_bal'
]


In [14]:
train_engineered_new=train_engineered_n.drop(columns=features_old)

In [15]:
train_engineered_new.shape

(40000, 203)

### 🏷️ Encoding Categorical Variables

We use **one-hot encoding** to convert categorical columns into numeric format so they can be used by the model.


In [16]:
# Identify the ID column, target column and feature columns
TARGET = "liquidity_stress_next_30d"
ID_COL = "ID"
object_type=['str','object']

feature_cols = [c for c in train_engineered_new.columns if c not in [ID_COL, TARGET]]
print(len(feature_cols))
cat_cols = [c for c in feature_cols if train_engineered_new[c].dtype in object_type]
num_cols = (
    train_engineered_new.select_dtypes(include=['number'])
    .drop(columns=[TARGET], errors='ignore')
    .columns.tolist()
)

cat_cols


201


['gender', 'region', 'smartphone', 'segment', 'earning_pattern']

In [17]:
## Inspect categorical columns for misspelling
for col in cat_cols:
  print(f"Unique values in {col}:", train[col].unique())

Unique values in gender: ['F' 'M']
Unique values in region: ['Central Zone' 'Northern Highlands' 'Lake Zone (Northwest)'
 'Coastal & Eastern Corridor' 'Southern Highlands' 'Islander'
 'Western Zone']
Unique values in smartphone: ['Yes' 'No']
Unique values in segment: ['MVC' 'LVC' 'HVC']
Unique values in earning_pattern: ['Monthly Earner' 'Weekly Earner' 'Irregular Earner']


In [18]:
for col in cat_cols:
  print(f"Unique values in {col}:", test[col].unique())

Unique values in gender: ['F' 'M']
Unique values in region: ['Central Zone' 'Northern Highlands' 'Lake Zone (Northwest)'
 'Coastal & Eastern Corridor' 'Southern Highlands' 'Islander'
 'Western Zone']
Unique values in smartphone: ['Yes' 'No']
Unique values in segment: ['MVC' 'LVC' 'HVC']
Unique values in earning_pattern: ['Monthly Earner' 'Weekly Earner' 'Irregular Earner']


In [19]:
## Split to Features and Target
X = train_engineered_new[feature_cols]

y=train_engineered_new[TARGET]

Split the training data to training and validation. We apply statify to ensure that equal proportion of the 1s and 0s are accounted for within each split

In [20]:
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

In [21]:
y_train.value_counts()

liquidity_stress_next_30d
0    23800
1     4200
Name: count, dtype: int64

In [22]:
y_val.value_counts()

liquidity_stress_next_30d
0    10200
1     1800
Name: count, dtype: int64

In [23]:
X_train.shape,X_val.shape

((28000, 201), (12000, 201))

We will now apply One Hot Encoding for the the categorical columns in the X_train and X_val

In [24]:
# 1. Initialize the encoder (drop='first' mimics what you used before)
ohe = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit and transform ONLY on the categorical training columns
train_encoded_array = ohe.fit_transform(X_train[cat_cols])
# 3. Transform ONLY on the categorical validation columns
val_encoded_array = ohe.transform(X_val[cat_cols])

# 4. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
X_train_encoded = pd.DataFrame(
    train_encoded_array,
    columns=encoded_feature_names,
    index=X_train.index
)

X_val_encoded = pd.DataFrame(
    val_encoded_array,
    columns=encoded_feature_names,
    index=X_val.index
)

X_train_encoded.head()


,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
39167,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
17329,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
20411,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
19056,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1001,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [25]:
# 3. Drop the old unencoded categorical columns from X_train and X_val
# and concatenate the new OHE columns alongside the numeric ones
X_train_ohe = pd.concat(
    [X_train.drop(columns=cat_cols), X_train_encoded],
    axis=1
)

X_val_ohe = pd.concat(
    [X_val.drop(columns=cat_cols), X_val_encoded],
    axis=1
)

X_train_ohe.head()

,arpu,age,x_90_d_activity_rate,paybill_volume_mean,paybill_volume_std,paybill_volume_ratio,paybill_volume_slope,paybill_volume_rel_slope,paybill_total_value_mean,paybill_total_value_std,paybill_total_value_ratio,paybill_total_value_slope,paybill_total_value_rel_slope,paybill_highest_amount_mean,paybill_highest_amount_std,paybill_highest_amount_ratio,paybill_companies_mean,paybill_companies_std,paybill_companies_ratio,merchantpay_volume_mean,merchantpay_volume_std,merchantpay_volume_ratio,merchantpay_volume_slope,merchantpay_volume_rel_slope,merchantpay_total_value_mean,merchantpay_total_value_std,merchantpay_total_value_ratio,merchantpay_total_value_slope,merchantpay_total_value_rel_slope,merchantpay_highest_amount_mean,merchantpay_highest_amount_std,merchantpay_highest_amount_ratio,merchantpay_merchants_mean,merchantpay_merchants_std,merchantpay_merchants_ratio,transfer_from_bank_volume_mean,transfer_from_bank_volume_std,transfer_from_bank_volume_ratio,transfer_from_bank_volume_slope,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,transfer_from_bank_total_value_ratio,transfer_from_bank_total_value_slope,transfer_from_bank_total_value_rel_slope,transfer_from_bank_highest_amount_mean,transfer_from_bank_highest_amount_std,transfer_from_bank_highest_amount_ratio,transfer_from_bank_banks_mean,transfer_from_bank_banks_std,transfer_from_bank_banks_ratio,mm_send_volume_mean,mm_send_volume_std,mm_send_volume_ratio,mm_send_volume_slope,mm_send_volume_rel_slope,mm_send_total_value_mean,mm_send_total_value_std,mm_send_total_value_ratio,mm_send_total_value_slope,mm_send_total_value_rel_slope,mm_send_highest_amount_mean,mm_send_highest_amount_std,mm_send_highest_amount_ratio,mm_send_recipients_mean,mm_send_recipients_std,mm_send_recipients_ratio,received_volume_mean,received_volume_std,received_volume_ratio,received_volume_slope,received_volume_rel_slope,received_total_value_mean,received_total_value_std,received_total_value_ratio,received_total_value_slope,received_total_value_rel_slope,received_highest_amount_mean,received_highest_amount_std,received_highest_amount_ratio,received_senders_mean,received_senders_std,received_senders_ratio,deposit_volume_mean,deposit_volume_std,deposit_volume_ratio,deposit_volume_slope,deposit_volume_rel_slope,deposit_total_value_mean,deposit_total_value_std,deposit_total_value_ratio,deposit_total_value_slope,deposit_total_value_rel_slope,deposit_highest_amount_mean,deposit_highest_amount_std,deposit_highest_amount_ratio,deposit_agents_mean,deposit_agents_std,deposit_agents_ratio,withdraw_volume_mean,...,withdraw_total_value_rel_slope,withdraw_highest_amount_mean,withdraw_highest_amount_std,withdraw_highest_amount_ratio,withdraw_agents_mean,withdraw_agents_std,withdraw_agents_ratio,bal_mean,bal_std,bal_ratio,bal_slope,bal_rel_slope,inflow_mean,inflow_std,inflow_ratio,inflow_slope,outflow_mean,outflow_std,outflow_ratio,outflow_slope,net_mean,net_std,net_slope,net_recent_minus_older,neg_net_months,neg_net_months_last3,out_in_ratio_6m,out_in_ratio_last3,inflow_cv,inflow_months,runway_m1,runway_mean,bal_vs_mean_outflow,bal_vs_max_withdrawal,cash_out_share,deposit_to_withdraw,paybill_share_outflow,paybill_active_months,paybill_active_last3,paybill_zero_streak,paybill_avg_ticket,paybill_spike,paybill_txn_per_counterparty,merchantpay_active_months,merchantpay_active_last3,merchantpay_zero_streak,merchantpay_avg_ticket,merchantpay_spike,merchantpay_txn_per_counterparty,transfer_from_bank_active_months,transfer_from_bank_active_last3,transfer_from_bank_zero_streak,transfer_from_bank_avg_ticket,transfer_from_bank_spike,transfer_from_bank_txn_per_counterparty,mm_send_active_months,mm_send_active_last3,mm_send_zero_streak,mm_send_avg_ticket,mm_send_spike,mm_send_txn_per_counterparty,received_active_months,received_active_last3,received_zero_streak,received_avg_ticket,received_spike,received_txn_per_counterparty,deposit_active_months,deposit_active_last3,deposit_

### Inspect for Feature Importance to the target variable using the correlation
We will apply correlation to see the linear relationship between each feature and the target

In [26]:
# 1. Calculate correlation with y_train and convert the resulting Series to a DataFrame
correlation_df = X_train_ohe.corrwith(y_train).to_frame(name='Correlation')

# 2. Sort by absolute correlation to see the most impactful features at the top
correlation_df = (
    correlation_df
    .assign(Abs_Correlation=correlation_df['Correlation'].abs())
    .sort_values(by='Abs_Correlation', ascending=False)
    .drop(columns=['Abs_Correlation'])
)

# Display the DataFrame
correlation_df

,Correlation
bal_rel_slope,-0.251231
withdraw_volume_ratio,0.224364
bal_ratio,-0.212569
neg_net_months_last3,0.206725
withdraw_agents_ratio,0.189933
...,...
region_Western Zone,0.001809
transfer_from_bank_total_value_ratio,0.001319
region_Lake Zone (Northwest),-0.001088
region_Northern Highlands,-0.000969


The Linear relationships are very weak this could mean the relationships are non-linear

### CHECK FOR COLLINEARITY (within the features)
We will check for collinearity amongst the numerical variables.

In [27]:
## Correlation function that find the linear relationship between features using a given threshold to identify collinear relationship

# def correlation(dataset, threshold):
#     col_corr = set()
#     col_pair = set()

#     corr_matrix = dataset.corr()

#     for i in range(len(corr_matrix.columns)):
#         for j in range(i):
#             if abs(corr_matrix.iloc[i, j]) > threshold:
#                 colname1 = corr_matrix.columns[i]
#                 colname2 = corr_matrix.columns[j]

#                 col_pair.add((colname1, colname2, corr_matrix.iloc[i, j]))
#                 col_corr.add(colname1)

#     return col_corr, col_pair

In [28]:
# col_corr, col_pair = correlation(X_train_ohe[num_cols], 0.7)
# col_pair

These are the collinear pairs. We will have to drop some of the pairs systematically in a way that we compare with the relationship with the target. 

In [29]:
# threshold = 0.7

# # Strength of each feature's relationship with the target
# target_corr = X_train_ohe[num_cols].corrwith(y_train).abs()

# # Pairwise feature correlations (upper triangle only, so each pair appears once)
# corr = X_train_ohe[num_cols].corr().abs()
# upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

# pairs = (
#     upper.stack()
#     .rename("pair_corr")
#     .reset_index()
#     .rename(columns={"level_0": "feat_a", "level_1": "feat_b"})
# )
# pairs = pairs[pairs["pair_corr"] > threshold].sort_values("pair_corr", ascending=False)

# # For each pair, drop the feature with the weaker target correlation
# to_drop, log = set(), []
# for _, r in pairs.iterrows():
#     a, b = r["feat_a"], r["feat_b"]
#     if a in to_drop or b in to_drop:      # pair already resolved by an earlier drop
#         continue
#     drop, keep = (a, b) if target_corr[a] < target_corr[b] else (b, a)
#     to_drop.add(drop)
#     log.append({
#         "dropped": drop,
#         "kept": keep,
#         "pair_corr": round(r["pair_corr"], 3),
#         "dropped_target_corr": round(target_corr[drop], 4),
#         "kept_target_corr": round(target_corr[keep], 4),
#         "gap": round(target_corr[keep] - target_corr[drop], 4),
#     })

# drop_log = pd.DataFrame(log)

# # Pairs where the two features are nearly tied on target correlation: review by hand
# close_calls = drop_log[drop_log["gap"] < 0.01]

# drop_log

In [30]:
# ## number of columns to drop
# len(to_drop)

In [31]:
# ## convert to list
# features_to_drop=list(to_drop)
# features_to_drop

## Deduction

Majority of the engineered columns are dropped when we used the pipeline we used for previous feature selection

In [32]:
# features_to_drop_prev=[
#  'm4_merchantpay_volume',
#  'm2_deposit_highest_amount',
#  'm2_merchantpay_volume',
#  'm5_deposit_volume',
#  'arpu',
#  'm1_paybill_volume',
#  'm3_paybill_volume',
#  'm1_deposit_highest_amount',
#  'm1_merchantpay_volume',
#  'm6_merchantpay_total_value',
#  'm6_merchantpay_volume',
#  'm1_mm_send_highest_amount',
#  'm5_paybill_volume',
#  'm4_withdraw_highest_amount',
#  'm1_received_highest_amount',
#  'm4_transfer_from_bank_volume',
#  'm6_withdraw_highest_amount',
#  'm5_withdraw_volume',
#  'm3_deposit_highest_amount',
#  'm4_merchantpay_highest_amount',
#  'm5_merchantpay_total_value',
#  'm5_transfer_from_bank_volume',
#  'm2_daily_avg_bal',
#  'm4_deposit_volume',
#  'm3_merchantpay_volume',
#  'm1_merchantpay_total_value',
#  'm2_withdraw_highest_amount',
#  'm1_withdraw_highest_amount',
#  'm6_mm_send_volume',
#  'm6_deposit_volume',
#  'm3_received_highest_amount',
#  'm3_transfer_from_bank_highest_amount',
#  'm2_paybill_volume',
#  'm5_merchantpay_volume',
#  'm2_merchantpay_highest_amount',
#  'm4_received_volume',
#  'm4_paybill_volume',
#  'm3_merchantpay_highest_amount',
#  'm2_transfer_from_bank_volume',
#  'm2_received_highest_amount',
#  'm1_transfer_from_bank_volume',
#  'm5_received_volume',
#  'm6_transfer_from_bank_volume',
#  'm6_received_volume',
#  'm3_mm_send_highest_amount',
#  'm3_daily_avg_bal',
#  'm2_mm_send_volume',
#  'm5_mm_send_volume',
#  'm3_withdraw_highest_amount',
#  'm4_mm_send_volume',
#  'm6_paybill_volume']

In [33]:
# len(features_to_drop_prev)

In [34]:
# X_train_ohe_adj=X_train_ohe.drop(columns=features_to_drop_prev)
# X_train_ohe_adj.head()

In [35]:
# X_val_ohe_adj=X_val_ohe.drop(columns=features_to_drop_prev)


In [36]:
# X_val_ohe_adj.shape, X_train_ohe_adj.shape

In [37]:
import numpy as np, pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

X = X_train_ohe.reset_index(drop=True)
y = y_train.reset_index(drop=True)
X_test = X_val_ohe.reset_index(drop=True)      # same columns, same order

SPLITS = 5
skf = StratifiedKFold(SPLITS, shuffle=True, random_state=42)   # fixed forever

lgb_params = {
    'colsample_bytree': 0.6521871356061031, 'max_depth': 6, 'min_child_samples': 64, 'min_split_gain': 0.0591267371675428, 'num_leaves': 20, 'reg_alpha': 0.9130691409658205, 'reg_lambda': 2.184404372168336, 'subsample': 0.7666039791481465, 'n_estimators':600,'learning_rate':0.03,'subsample_freq':1,'n_jobs':-1,'verbose':-1,
}

cat_params = {'border_count': 128, 'depth': 5, 'l2_leaf_reg': np.float64(5.477661047824163), 'random_strength': np.float64(0.5491734552400583), 'rsm': np.float64(0.7211423872974992), 'subsample': np.float64(0.7799016533479063),'iterations':500, 'learning_rate':0.06,'bootstrap_type':"Bernoulli",'loss_function':"Logloss", 'thread_count':-1,'verbose':0, 'allow_writing_files':False}

xgb_params = {'colsample_bytree': np.float64(0.7037200775820313), 'gamma': np.float64(1.7921825998469865), 'max_depth': 3, 'min_child_weight': 1, 'reg_alpha': np.float64(0.0069682757636133075), 'reg_lambda': np.float64(0.9611426142207995), 'subsample': np.float64(0.9272059063689972), 'n_estimators':400, 'learning_rate':0.05,'tree_method':"hist",'eval_metric':"logloss",'n_jobs':-1
}



def run_cv(kind, params, X, y, X_test, seed=42):
    oof = np.zeros(len(X)); val_prob = np.zeros(len(X_test))
    imp = pd.DataFrame(index=X.columns); best_iters = []
    models=[]

    for k, (tr, va) in enumerate(skf.split(X, y)):
        Xtr, Xva, ytr, yva = X.iloc[tr], X.iloc[va], y.iloc[tr], y.iloc[va]

        if kind == "lgb":
            m = lgb.LGBMClassifier(**params,random_state=seed)
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="binary_logloss",
                  callbacks=[lgb.early_stopping(200, verbose=False)])
            best_iters.append(m.best_iteration_)
            imp[k] = m.booster_.feature_importance("gain")
        elif kind == "cat":
            m = CatBoostClassifier(**params, random_seed=seed)
            m.fit(Xtr, ytr, eval_set=(Xva, yva))
            best_iters.append(m.get_best_iteration())
            imp[k] = m.get_feature_importance()
        elif kind == "xgb":
            m = XGBClassifier(**params, random_state=seed)
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            try:
                best_iters.append(m.best_iteration)
            except AttributeError:                       # early stopping was not used
                best_iters.append(m.get_params()["n_estimators"])
            imp[k] = m.feature_importances_
            
        models.append(m)  
        oof[va] = m.predict_proba(Xva)[:, 1]
        val_prob += m.predict_proba(X_test)[:, 1] / SPLITS   # fold-averaged

    print(f"{kind} seed={seed}  logloss={log_loss(y, oof):.5f}  "
          f"auc={roc_auc_score(y, oof):.5f}  iters={int(np.mean(best_iters))}")
    
    val_pred = (val_prob >= 0.5).astype(int)
    train_pred = (oof >= 0.5).astype(int)

    train_auc = roc_auc_score(y, oof)
    val_auc = roc_auc_score(y_val, val_prob)
    # rep = classification_report(y_val, val_pred, output_dict=True)

    print(f"Train ROC-AUC: {train_auc} | Validation ROC-AUC: {val_auc}")
    print(f"Train accuracy: {accuracy_score(y, train_pred)} | "
            f"Validation accuracy: {accuracy_score(y_val, val_pred)}")
    ## Log-loss for the predicted classes
    print(f"Train prediction log_loss: {log_loss(y, train_pred)} | "
            f"Validation prediction log_loss: {log_loss(y_val, val_pred)}")
    ## Log-loss for the predicted probability
    print(f"Train probability log loss: {log_loss(y, oof)} | Val probability log loss: {log_loss(y_val, val_prob)}")

    print(classification_report(y_val, val_pred))
    print(confusion_matrix(y_val, val_pred))
    print("\n" + "-" * 60 + "\n")

    return dict(oof=oof, test=val_prob, imp=imp, iters=best_iters, models=models)

In [38]:
class Blend:
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights or [1 / len(models)] * len(models)

    def predict_proba(self, X):
        p = sum(w * m.predict_proba(X)[:, 1] for w, m in zip(self.weights, self.models))
        return np.column_stack([1 - p, p])

In [39]:
res = {}
for s in [1, 2, 3]:
    res[f"lgb_s{s}"] = run_cv("lgb", lgb_params, X, y, X_test, seed=s)
res["cat"] = run_cv("cat", cat_params, X, y, X_test, seed=1)
res["xgb"] = run_cv("xgb", xgb_params, X, y, X_test, seed=1)

all_models = []
for s in [1, 2, 3]:
    all_models += res[f"lgb_s{s}"]["models"]
all_models += res["cat"]["models"]
all_models += res["xgb"]["models"]


# average the LightGBM seeds into one model, then blend the three families
lgb_oof  = np.mean([res[f"lgb_s{s}"]["oof"]  for s in [1, 2, 3]], axis=0)
lgb_test = np.mean([res[f"lgb_s{s}"]["test"] for s in [1, 2, 3]], axis=0)

blend_oof  = (lgb_oof  + res["cat"]["oof"]  + res["xgb"]["oof"])  / 3
blend_test = (lgb_test + res["cat"]["test"] + res["xgb"]["test"]) / 3

print("blend logloss:", log_loss(y, blend_oof), " auc:", roc_auc_score(y, blend_oof))

c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

lgb seed=1  logloss=0.23515  auc=0.91751  iters=323
Train ROC-AUC: 0.9175087134853942 | Validation ROC-AUC: 0.9149529956427016
Train accuracy: 0.9036785714285714 | Validation accuracy: 0.9026666666666666
Train prediction log_loss: 3.4717761853731774 | Validation prediction log_loss: 3.5082489298740693
Train probability log loss: 0.2351480633957751 | Val probability log loss: 0.23877625937749744
              precision    recall  f1-score   support

           0       0.92      0.97      0.94     10200
           1       0.77      0.50      0.61      1800

    accuracy                           0.90     12000
   macro avg       0.84      0.74      0.78     12000
weighted avg       0.89      0.90      0.89     12000

[[9931  269]
 [ 899  901]]

------------------------------------------------------------



c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

lgb seed=2  logloss=0.23534  auc=0.91745  iters=317
Train ROC-AUC: 0.9174503601440577 | Validation ROC-AUC: 0.9146688997821351
Train accuracy: 0.9038571428571428 | Validation accuracy: 0.90225
Train prediction log_loss: 3.465339818696549 | Validation prediction log_loss: 3.5232671187862024
Train probability log loss: 0.2353366061876452 | Val probability log loss: 0.2390887719259003
              precision    recall  f1-score   support

           0       0.92      0.97      0.94     10200
           1       0.77      0.50      0.61      1800

    accuracy                           0.90     12000
   macro avg       0.84      0.74      0.78     12000
weighted avg       0.89      0.90      0.89     12000

[[9924  276]
 [ 897  903]]

------------------------------------------------------------



c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

lgb seed=3  logloss=0.23575  auc=0.91690  iters=322
Train ROC-AUC: 0.9169018707482992 | Validation ROC-AUC: 0.9150818627450981
Train accuracy: 0.9037142857142857 | Validation accuracy: 0.903
Train prediction log_loss: 3.4704889120378515 | Validation prediction log_loss: 3.496234378744364
Train probability log loss: 0.23574851060706217 | Val probability log loss: 0.2385838996089858
              precision    recall  f1-score   support

           0       0.92      0.97      0.94     10200
           1       0.77      0.51      0.61      1800

    accuracy                           0.90     12000
   macro avg       0.84      0.74      0.78     12000
weighted avg       0.90      0.90      0.89     12000

[[9927  273]
 [ 891  909]]

------------------------------------------------------------

cat seed=1  logloss=0.23279  auc=0.91904  iters=340
Train ROC-AUC: 0.919035894357743 | Validation ROC-AUC: 0.9168672657952069
Train accuracy: 0.9050357142857143 | Validation accuracy: 0.9034166666666

In [41]:
val_pred = (blend_test >= 0.5).astype(int) ## recall blend_test is in predict_proba
print("Val blend logloss:", log_loss(y_val, blend_test), " Val auc:", roc_auc_score(y_val, blend_test), "Val accuracy:", {accuracy_score(y_val, val_pred)})

print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))
print("\n" + "-" * 60 + "\n")

Val blend logloss: 0.23704369142191362  Val auc: 0.9162636165577341 Val accuracy: {0.9033333333333333}
              precision    recall  f1-score   support

           0       0.92      0.97      0.94     10200
           1       0.77      0.50      0.61      1800

    accuracy                           0.90     12000
   macro avg       0.85      0.74      0.78     12000
weighted avg       0.90      0.90      0.89     12000

[[9936  264]
 [ 896  904]]

------------------------------------------------------------



In [42]:
imp_all = pd.concat([res[f"lgb_s{s}"]["imp"] for s in [1, 2, 3]], axis=1).mean(axis=1)
imp_all = imp_all.sort_values(ascending=False)

In [43]:
imp_all.head(20)

bal_ratio                           26692.800468
out_in_ratio_last3                  25275.096376
inflow_ratio                        17902.457213
withdraw_total_value_ratio          10766.506585
bal_rel_slope                        8819.024690
withdraw_highest_amount_ratio        3404.906301
bal_std                              3197.068867
paybill_total_value_ratio            2319.933878
bal_slope                            2234.420461
withdraw_volume_ratio                2124.103498
net_recent_minus_older               1735.336997
inflow_slope                         1472.228862
merchantpay_highest_amount_ratio     1256.265872
x_90_d_activity_rate                 1234.742327
out_in_ratio_6m                      1206.361226
deposit_total_value_ratio            1022.229894
merchantpay_total_value_ratio         994.025760
paybill_volume_ratio                  946.808930
merchantpay_merchants_ratio           924.483308
paybill_highest_amount_ratio          842.278632
dtype: float64

In [57]:
lgb_models = [m for s in [1, 2, 3] for m in res[f"lgb_s{s}"]["models"]]   # 15 models
cat_models = res["cat"]["models"]                                          # 5
xgb_models = res["xgb"]["models"]                                          # 5

models  = lgb_models + cat_models + xgb_models
weights = ([1/3/len(lgb_models)] * len(lgb_models) +
           [1/3/len(cat_models)] * len(cat_models) +
           [1/3/len(xgb_models)] * len(xgb_models))

blend = Blend(models, weights)

In [ ]:
# lgbm_best_parameters={'colsample_bytree': 0.6521871356061031, 'max_depth': 6, 'min_child_samples': 64, 'min_split_gain': 0.0591267371675428, 'num_leaves': 20, 'reg_alpha': 0.9130691409658205, 'reg_lambda': 2.184404372168336, 'subsample': 0.7666039791481465}

# xgb_best_parameters={'colsample_bytree': np.float64(0.7037200775820313), 'gamma': np.float64(1.7921825998469865), 'max_depth': 3, 'min_child_weight': 1, 'reg_alpha': np.float64(0.0069682757636133075), 'reg_lambda': np.float64(0.9611426142207995), 'subsample': np.float64(0.9272059063689972)}

# catboost_best_parameters={'border_count': 128, 'depth': 5, 'l2_leaf_reg': np.float64(5.477661047824163), 'random_strength': np.float64(0.5491734552400583), 'rsm': np.float64(0.7211423872974992), 'subsample': np.float64(0.7799016533479063)}



## Create submission 

In [58]:
SUBMISSION_TARGET = "Target"

# Create a sample submission template directly from the test IDs
submission = pd.DataFrame({
    "ID": test["ID"],
    SUBMISSION_TARGET: 0.5
})


In [59]:
test_clean=test_engineered_n.drop(columns=['ID'],inplace=True)

In [60]:
test_clean=test_engineered_n.drop(columns=features_old)

In [61]:
# Apply one hot encoding
# 2. Fit transform ONLY on the categorical training columns
test_encoded_array = ohe.transform(test_clean[cat_cols])

# 3. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
test_encoded = pd.DataFrame(
    test_encoded_array,
    columns=encoded_feature_names,
    index=test_clean.index
)

test_encoded.head()

,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [62]:
test_ohe = pd.concat(
    [test_clean.drop(columns=cat_cols), test_encoded],
    axis=1
)

test_ohe.head()

,arpu,age,x_90_d_activity_rate,paybill_volume_mean,paybill_volume_std,paybill_volume_ratio,paybill_volume_slope,paybill_volume_rel_slope,paybill_total_value_mean,paybill_total_value_std,paybill_total_value_ratio,paybill_total_value_slope,paybill_total_value_rel_slope,paybill_highest_amount_mean,paybill_highest_amount_std,paybill_highest_amount_ratio,paybill_companies_mean,paybill_companies_std,paybill_companies_ratio,merchantpay_volume_mean,merchantpay_volume_std,merchantpay_volume_ratio,merchantpay_volume_slope,merchantpay_volume_rel_slope,merchantpay_total_value_mean,merchantpay_total_value_std,merchantpay_total_value_ratio,merchantpay_total_value_slope,merchantpay_total_value_rel_slope,merchantpay_highest_amount_mean,merchantpay_highest_amount_std,merchantpay_highest_amount_ratio,merchantpay_merchants_mean,merchantpay_merchants_std,merchantpay_merchants_ratio,transfer_from_bank_volume_mean,transfer_from_bank_volume_std,transfer_from_bank_volume_ratio,transfer_from_bank_volume_slope,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,transfer_from_bank_total_value_ratio,transfer_from_bank_total_value_slope,transfer_from_bank_total_value_rel_slope,transfer_from_bank_highest_amount_mean,transfer_from_bank_highest_amount_std,transfer_from_bank_highest_amount_ratio,transfer_from_bank_banks_mean,transfer_from_bank_banks_std,transfer_from_bank_banks_ratio,mm_send_volume_mean,mm_send_volume_std,mm_send_volume_ratio,mm_send_volume_slope,mm_send_volume_rel_slope,mm_send_total_value_mean,mm_send_total_value_std,mm_send_total_value_ratio,mm_send_total_value_slope,mm_send_total_value_rel_slope,mm_send_highest_amount_mean,mm_send_highest_amount_std,mm_send_highest_amount_ratio,mm_send_recipients_mean,mm_send_recipients_std,mm_send_recipients_ratio,received_volume_mean,received_volume_std,received_volume_ratio,received_volume_slope,received_volume_rel_slope,received_total_value_mean,received_total_value_std,received_total_value_ratio,received_total_value_slope,received_total_value_rel_slope,received_highest_amount_mean,received_highest_amount_std,received_highest_amount_ratio,received_senders_mean,received_senders_std,received_senders_ratio,deposit_volume_mean,deposit_volume_std,deposit_volume_ratio,deposit_volume_slope,deposit_volume_rel_slope,deposit_total_value_mean,deposit_total_value_std,deposit_total_value_ratio,deposit_total_value_slope,deposit_total_value_rel_slope,deposit_highest_amount_mean,deposit_highest_amount_std,deposit_highest_amount_ratio,deposit_agents_mean,deposit_agents_std,deposit_agents_ratio,withdraw_volume_mean,...,withdraw_total_value_rel_slope,withdraw_highest_amount_mean,withdraw_highest_amount_std,withdraw_highest_amount_ratio,withdraw_agents_mean,withdraw_agents_std,withdraw_agents_ratio,bal_mean,bal_std,bal_ratio,bal_slope,bal_rel_slope,inflow_mean,inflow_std,inflow_ratio,inflow_slope,outflow_mean,outflow_std,outflow_ratio,outflow_slope,net_mean,net_std,net_slope,net_recent_minus_older,neg_net_months,neg_net_months_last3,out_in_ratio_6m,out_in_ratio_last3,inflow_cv,inflow_months,runway_m1,runway_mean,bal_vs_mean_outflow,bal_vs_max_withdrawal,cash_out_share,deposit_to_withdraw,paybill_share_outflow,paybill_active_months,paybill_active_last3,paybill_zero_streak,paybill_avg_ticket,paybill_spike,paybill_txn_per_counterparty,merchantpay_active_months,merchantpay_active_last3,merchantpay_zero_streak,merchantpay_avg_ticket,merchantpay_spike,merchantpay_txn_per_counterparty,transfer_from_bank_active_months,transfer_from_bank_active_last3,transfer_from_bank_zero_streak,transfer_from_bank_avg_ticket,transfer_from_bank_spike,transfer_from_bank_txn_per_counterparty,mm_send_active_months,mm_send_active_last3,mm_send_zero_streak,mm_send_avg_ticket,mm_send_spike,mm_send_txn_per_counterparty,received_active_months,received_active_last3,received_zero_streak,received_avg_ticket,received_spike,received_txn_per_counterparty,deposit_active_months,deposit_active_last3,deposit_

In [52]:
# # Adjust the number of features
# test_ohe_adj=test_ohe.drop(columns=features_to_drop_prev)
# test_ohe_adj.head()

In [63]:
test_prob=blend.predict_proba(test_ohe)[:, 1]

In [64]:
submission[SUBMISSION_TARGET] = test_prob

In [65]:
submission

,ID,Target
0,ID_ED91AC7A28,0.019153
1,ID_29975A9C7E,0.048962
2,ID_F59CE25650,0.017472
3,ID_E875F68D1A,0.005417
4,ID_2902B1C0D0,0.005284
...,...,...
29995,ID_3C7CF79AEE,0.008604
29996,ID_F1651E73C4,0.015731
29997,ID_FD3F0934CA,0.237549
29998,ID_DA6916D6C7,0.008382


just a quick inspection

In [66]:
classification = (test_prob >= 0.5).astype(int)

In [67]:
classification

array([0, 0, 0, ..., 0, 0, 0])

In [68]:
from collections import Counter
class_freq=Counter(classification)
class_freq

Counter({0: 26914, 1: 3086})

In [69]:
# Save to CSV
submission.to_csv("submission_Ensemble.csv", index=False)
print("Submission file saved!")


Submission file saved!
